# KPI Dictionary & Data Quality Profile
This notebook uses the supplied `sales_data.csv` file. It checks completeness, uniqueness, validity, consistency and freshness, then calculates the KPIs.

In [1]:
import pandas as pd
df=pd.read_csv('data/sales_data.csv')
df['Date']=pd.to_datetime(df['Date'],errors='coerce')
display(df)
print('Rows:',len(df))

,Date,Product,Category,Region,Units,Revenue
0,2026-01-05,Laptop Pro 14,Electronics,North,18,216000
1,2026-01-12,Wireless Headphones,Electronics,South,42,126000


Rows: 2


## Completeness

In [2]:
missing=df.isna().sum()
display(pd.DataFrame({'Missing Values':missing,'Missing %':(missing/len(df)*100).round(2)}))
print('Overall completeness:',round((1-df.isna().sum().sum()/(df.shape[0]*df.shape[1]))*100,2),'%')

,Missing Values,Missing %
Date,0,0.0
Product,0,0.0
Category,0,0.0
Region,0,0.0
Units,0,0.0
Revenue,0,0.0


Overall completeness: 100.0 %


## Uniqueness

In [3]:
dupes=int(df.duplicated().sum())
print('Duplicate rows:',dupes)
print('Duplicate rate:',round(dupes/len(df)*100,2),'%')

Duplicate rows: 0
Duplicate rate: 0.0 %


## Validity

In [4]:
print('Invalid dates:',int(df['Date'].isna().sum()))
print('Non-positive Units:',int((df['Units']<=0).sum()))
print('Negative Revenue:',int((df['Revenue']<0).sum()))

Invalid dates: 0
Non-positive Units: 0
Negative Revenue: 0


## Consistency

In [5]:
for col in ['Product','Category','Region']:
    print(col,'blank values:',int(df[col].astype(str).str.strip().eq('').sum()))
print('Categories:',df['Category'].dropna().unique().tolist())
print('Regions:',df['Region'].dropna().unique().tolist())

Product blank values: 0
Category blank values: 0
Region blank values: 0
Categories: ['Electronics']
Regions: ['North', 'South']


## Freshness

In [6]:
reference=pd.Timestamp('2026-09-17')
latest=df['Date'].max()
age=(reference-latest).days
print('Latest record:',latest.date())
print('Reference date:',reference.date())
print('Age in days:',age)
print('Daily freshness:', 'PASS' if age<=1 else 'FAIL')

Latest record: 2026-01-12
Reference date: 2026-09-17
Age in days: 248
Daily freshness: FAIL


## KPI Calculations

In [7]:
total_revenue=df['Revenue'].sum()
total_units=df['Units'].sum()
records=len(df)
kpis=pd.DataFrame({'KPI':['Total Revenue','Total Units Sold','Number of Sales Records','Average Revenue per Record','Average Revenue per Unit','Average Units per Record','Number of Products','Number of Regions'],'Value':[total_revenue,total_units,records,round(total_revenue/records,2),round(total_revenue/total_units,2),round(total_units/records,2),df['Product'].nunique(),df['Region'].nunique()]})
display(kpis)

,KPI,Value
0,Total Revenue,342000.0
1,Total Units Sold,60.0
2,Number of Sales Records,2.0
3,Average Revenue per Record,171000.0
4,Average Revenue per Unit,5700.0
5,Average Units per Record,30.0
6,Number of Products,2.0
7,Number of Regions,2.0


## Contract Status

In [8]:
status={
 'Completeness':df.isna().sum().sum()==0,
 'Uniqueness':df.duplicated().sum()==0,
 'Validity':df['Date'].notna().all() and (df['Units']>0).all() and (df['Revenue']>=0).all(),
 'Consistency':all(df[c].astype(str).str.strip().ne('').all() for c in ['Product','Category','Region']),
 'Freshness (daily)':age<=1}
for k,v in status.items(): print(k,':','PASS' if v else 'FAIL')

Completeness : PASS
Uniqueness : PASS
Validity : PASS
Consistency : PASS
Freshness (daily) : FAIL
